# Gate 0B - Controlled Raw Acquisition

This notebook performs the first controlled raw-data acquisition for Phase 1.

It acquires only the smallest defensible first files:

- an OpenStreetMap walk graph inside the Gate 0 boundary;
- the OS Open Greenspace `SP` shapefile zip for pavilion/search-area context.

It does **not** score routes, rank interventions, or make funding-facing claims.

## Acquisition Rules

- Every raw file must be recorded in `raw_data_manifest_phase1.csv`.
- Every raw file must have a SHA-256 checksum.
- Source-published MD5 checks must be verified where available.
- Source acquisition status can advance to `raw_acquired_pending_quality_audit`, but quality, cross-check, and funding gates remain closed.
- These files are raw evidence inputs, not conclusions.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import geopandas as gpd
import osmnx as ox
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PHASE1_ROOT = PROJECT_ROOT.parent
else:
    PHASE1_ROOT = PROJECT_ROOT / "phase1_spinelens_ai"

sys.path.insert(0, str(PHASE1_ROOT / "src"))

from spinelens.gate0b import (  # noqa: E402
    build_manifest_row,
    download_file_with_checks,
    mark_sources_raw_acquired,
    raw_data_manifest_fieldnames,
    read_csv_rows,
    sha256_file,
    upsert_manifest_rows,
    utc_now_iso,
    write_csv_rows,
)

DATA = PHASE1_ROOT / "data"
RAW = DATA / "raw"
INTERIM = DATA / "interim"
REPORTS = PHASE1_ROOT / "outputs" / "reports"
RAW.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

BOUNDARY_PATH = INTERIM / "study_area_boundary_phase1.geojson"
OS_CANDIDATES_PATH = INTERIM / "gate0b_os_download_candidates.csv"
MANIFEST_PATH = DATA / "raw_data_manifest_phase1.csv"
ACQUISITION_PATH = DATA / "source_acquisition_status_phase1.csv"
ACQUISITION_NOTE_PATH = REPORTS / "gate0b_controlled_raw_acquisition_note.md"

manifest_rows = read_csv_rows(MANIFEST_PATH)
acquisition_rows = read_csv_rows(ACQUISITION_PATH)
pd.DataFrame(acquisition_rows).query("source_id in ['osm_network', 'os_open_greenspace']")

## Load Gate 0 Boundary

The OSM graph is extracted only inside the provisional Gate 0 boundary. This keeps the first acquisition small and relevant.

In [ ]:
boundary_gdf = gpd.read_file(BOUNDARY_PATH).to_crs(4326)
boundary_polygon = boundary_gdf.geometry.iloc[0]

boundary_summary = {
    "boundary_path": str(BOUNDARY_PATH.relative_to(PHASE1_ROOT)),
    "crs": str(boundary_gdf.crs),
    "geometry_type": boundary_polygon.geom_type,
    "bounds": tuple(round(value, 6) for value in boundary_polygon.bounds),
}
boundary_summary

## Acquire OSM Walk Graph

This is the first raw pedestrian-network extract. It is volunteered data and must be validated before it can support design or funding claims.

In [ ]:
osm_raw_dir = RAW / "osm_network"
osm_raw_dir.mkdir(parents=True, exist_ok=True)

ox.settings.use_cache = True
ox.settings.cache_folder = str(osm_raw_dir / "osmnx_cache")
ox.settings.requests_timeout = 180
ox.settings.overpass_rate_limit = True

osm_graph = ox.graph_from_polygon(
    boundary_polygon,
    network_type="walk",
    simplify=True,
    retain_all=True,
    truncate_by_edge=True,
)

osm_graph_path = osm_raw_dir / "gate0b_osm_walk_graph.graphml"
ox.save_graphml(osm_graph, filepath=osm_graph_path)

osm_nodes, osm_edges = ox.graph_to_gdfs(osm_graph)
osm_summary = {
    "raw_file": str(osm_graph_path.relative_to(PHASE1_ROOT)).replace("\\", "/"),
    "nodes": int(len(osm_nodes)),
    "edges": int(len(osm_edges)),
    "file_size_bytes": osm_graph_path.stat().st_size,
    "sha256": sha256_file(osm_graph_path),
}
osm_summary

## Acquire OS Open Greenspace SP

This is a small authoritative context layer for checking the Ryder Street grassland/pavilion hypothesis. It is not a land-ownership or buildability conclusion.

In [ ]:
os_candidates = pd.read_csv(OS_CANDIDATES_PATH)
greenspace_candidates = os_candidates[
    (os_candidates["source_id"] == "os_open_greenspace")
    & (os_candidates["area"] == "SP")
    & (os_candidates["format"] == "ESRI® Shapefile")
    & (os_candidates["download_decision"] == "safe_small_candidate_for_pavilion_context")
]

if len(greenspace_candidates) != 1:
    raise ValueError(f"Expected one OS Open Greenspace SP shapefile candidate, found {len(greenspace_candidates)}")

greenspace_candidate = greenspace_candidates.iloc[0].to_dict()
greenspace_raw_dir = RAW / "os_open_greenspace"
greenspace_path = greenspace_raw_dir / greenspace_candidate["file_name"]

greenspace_download = download_file_with_checks(
    url=greenspace_candidate["download_url"],
    output_path=greenspace_path,
    expected_md5=greenspace_candidate["md5"],
    max_bytes=10_000_000,
    timeout_seconds=90,
)

greenspace_summary = {
    "raw_file": str(greenspace_path.relative_to(PHASE1_ROOT)).replace("\\", "/"),
    "file_size_bytes": greenspace_download.file_size_bytes,
    "sha256": greenspace_download.sha256,
    "source_md5": greenspace_candidate["md5"],
    "local_md5": greenspace_download.md5,
}
greenspace_summary

## Update Manifest And Acquisition Ledger

The manifest records raw files. The acquisition ledger advances only to raw-acquired/pending-quality-audit.

In [ ]:
new_manifest_rows = [
    build_manifest_row(
        manifest_id="gate0b_osm_walk_graph_boundary_v0",
        source_id="osm_network",
        raw_file_path=osm_graph_path,
        phase1_root=PHASE1_ROOT,
        download_url="https://overpass-api.de/api/interpreter",
        source_publication_date="continuous",
        source_version="OSMnx walk-network extract inside Gate 0 boundary",
        license_name="ODbL",
        provenance_note=(
            f"Extracted with OSMnx from Gate 0 boundary; "
            f"nodes={osm_summary['nodes']}; edges={osm_summary['edges']}; "
            "volunteered data requiring validation."
        ),
    ),
    build_manifest_row(
        manifest_id="gate0b_os_open_greenspace_sp_shapefile",
        source_id="os_open_greenspace",
        raw_file_path=greenspace_path,
        phase1_root=PHASE1_ROOT,
        download_url=greenspace_candidate["download_url"],
        source_publication_date="October 2025 stated in source registry",
        source_version="OpenGreenspace SP ESRI Shapefile",
        license_name="OGL",
        provenance_note=(
            f"Downloaded from OS Downloads API metadata candidate; "
            f"source_md5={greenspace_candidate['md5']}; local_md5={greenspace_download.md5}."
        ),
    ),
]

updated_manifest_rows = upsert_manifest_rows(manifest_rows, new_manifest_rows)
write_csv_rows(MANIFEST_PATH, updated_manifest_rows, raw_data_manifest_fieldnames())

updated_acquisition_rows = mark_sources_raw_acquired(
    acquisition_rows,
    {"osm_network", "os_open_greenspace"},
)
write_csv_rows(ACQUISITION_PATH, updated_acquisition_rows, list(acquisition_rows[0].keys()))

pd.DataFrame(updated_manifest_rows)

## Acquisition Note

This note is the review checkpoint before any quality audit, graph scoring, tactical-corridor scoring, or digital wayfinder content generation.

In [ ]:
checked_at = utc_now_iso()
note = "\n".join([
    "# Gate 0B Controlled Raw Acquisition Note",
    "",
    f"Generated: {checked_at}",
    "",
    "## Decision",
    "",
    "The first controlled raw acquisition is complete. The project now has an OSM walk-network extract and a small OS Open Greenspace SP raw file. These are inputs for quality audit, not route recommendations.",
    "",
    "## Acquired Files",
    "",
    "| Source | File | Bytes | SHA-256 |",
    "|---|---|---:|---|",
    f"| OSM walk network | {osm_summary['raw_file']} | {osm_summary['file_size_bytes']} | {osm_summary['sha256']} |",
    f"| OS Open Greenspace SP | {greenspace_summary['raw_file']} | {greenspace_summary['file_size_bytes']} | {greenspace_summary['sha256']} |",
    "",
    "## OSM Graph Summary",
    "",
    f"- Nodes: {osm_summary['nodes']}",
    f"- Edges: {osm_summary['edges']}",
    "- Scope: Gate 0 provisional boundary only.",
    "- Caution: OSM is volunteered data and must be cross-checked before use in funding claims.",
    "",
    "## Gate Status",
    "",
    "Raw acquisition has advanced for `osm_network` and `os_open_greenspace`, but both remain pending quality audit. No source is funding-ready.",
    "",
    "## Next Controlled Step",
    "",
    "Run a quality audit on the OSM graph and OS Greenspace file: inspect CRS, geometry validity, graph connectivity, route-origin snapping, candidate pavilion context, and obvious missing pedestrian links. Only after that should route-family experiments begin.",
])

ACQUISITION_NOTE_PATH.write_text(note, encoding="utf-8")
print(note)